In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

"""
完全复制原始brain_voxel.ipynb的训练逻辑
使用现有的MAT数据加载方法，但严格按照原始notebook的数据分割和训练方式
"""

import os
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, accuracy_score, cohen_kappa_score
from torch.utils.data import Dataset, DataLoader
import time
import sys

# 导入你现有的MAT加载函数
sys.path.append('.')
from data.mat_loader import load_mat_data

class ExactReplicaDataset(Dataset):
    """
    完全复制原始notebook逻辑的数据集类
    """
    def __init__(self, data, labels):
        self.data = torch.FloatTensor(data)
        self.labels = torch.FloatTensor(labels)  # 保持one-hot格式，和原始notebook一致
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]

class ReplicaMLPModel(nn.Module):
    """
    完全复制原始notebook的模型架构：4x4096 + Dropout + L2正则化
    """
    def __init__(self, input_dim=341, num_classes=102, dropout_rate=0.5, l2_reg=1e-5):
        super(ReplicaMLPModel, self).__init__()
        
        self.layers = nn.Sequential(
            # 第一层：341 -> 4096
            nn.Linear(input_dim, 4096),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            # 第二层：4096 -> 4096  
            nn.Linear(4096, 4096),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            # 第三层：4096 -> 4096
            nn.Linear(4096, 4096), 
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            # 第四层：4096 -> 4096
            nn.Linear(4096, 4096),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            # 输出层：4096 -> 102
            nn.Linear(4096, num_classes),
            nn.Softmax(dim=1)
        )
        
        # 应用L2正则化到所有线性层
        self.l2_reg = l2_reg
        for layer in self.layers:
            if isinstance(layer, nn.Linear):
                nn.init.xavier_uniform_(layer.weight)
                if layer.bias is not None:
                    nn.init.zeros_(layer.bias)
    
    def forward(self, x):
        return self.layers(x)
    
    def get_l2_loss(self):
        """计算L2正则化损失"""
        l2_loss = 0
        for layer in self.layers:
            if isinstance(layer, nn.Linear):
                l2_loss += torch.sum(layer.weight ** 2)
        return self.l2_reg * l2_loss

def load_data_exact_replica(mat_file_path, random_state=42):
    """
    完全复制原始notebook的数据加载和处理逻辑
    """
    print("=== 完全复制原始notebook的数据处理 ===")
    
    # 使用你现有的MAT加载函数
    print(f"加载MAT文件: {mat_file_path}")
    arrays = load_mat_data(mat_file_path)
    
    # 转置数据（和原始notebook一致）
    train_data = arrays['data'].transpose() if 'data' in arrays else arrays['data'].T
    train_region = arrays['region'].transpose() if 'region' in arrays else arrays['region'].T
    prob_idx = arrays['prob_idx'].transpose().flatten() if 'prob_idx' in arrays else arrays['prob_idx'].flatten()
    
    print(f"原始数据形状:")
    print(f"  train_data: {train_data.shape}")
    print(f"  train_region: {train_region.shape}")
    print(f"  prob_idx: {prob_idx.shape}")
    print(f"  患者ID范围: {np.min(prob_idx)} - {np.max(prob_idx)}")
    
    # 第一步：按患者分割（完全复制notebook逻辑）
    curr_set = np.where(prob_idx != 38)[0]  # 患者1-37
    set_data = train_data[curr_set, :]
    set_region = train_region[curr_set, :]
    print(f"患者1-37数据形状: {set_data.shape}")
    
    curr_val = np.where(prob_idx == 38)[0]   # 患者38
    val_data = train_data[curr_val, :]
    val_label = train_region[curr_val, :]
    print(f"患者38数据形状: {val_data.shape}")
    
    # 第二步：从患者1-37中分出1%作为测试集（完全复制notebook）
    X_train1, X_train2, y_train1, y_train2 = train_test_split(
        set_data, set_region, 
        test_size=0.01,  # 和原始notebook完全一致
        random_state=42  # 和原始notebook完全一致
    )
    
    print(f"数据分割结果:")
    print(f"  训练集: {X_train1.shape} (患者1-37的99%)")
    print(f"  测试集: {X_train2.shape} (患者1-37的1%)")
    print(f"  验证集: {val_data.shape} (患者38的100%)")
    
    # 第三步：标准化（只基于训练集，完全复制notebook）
    print("应用标准化...")
    scaler = StandardScaler()
    scaler.fit(X_train1)  # 只基于训练集拟合
    
    X_train1_scaled = scaler.transform(X_train1)
    X_train2_scaled = scaler.transform(X_train2)  # 测试集
    val_data_scaled = scaler.transform(val_data)   # 验证集
    
    print("标准化完成")
    
    # 检查标签格式
    print(f"标签格式检查:")
    print(f"  训练标签形状: {y_train1.shape}")
    print(f"  标签和: {np.sum(y_train1[0])}")  # one-hot应该和为1
    print(f"  标签范围: {np.min(y_train1)} - {np.max(y_train1)}")
    
    return {
        'train_data': X_train1_scaled,
        'train_labels': y_train1,
        'test_data': X_train2_scaled, 
        'test_labels': y_train2,
        'val_data': val_data_scaled,
        'val_labels': val_label,
        'scaler': scaler,
        'feature_dim': X_train1_scaled.shape[1],
        'num_classes': y_train1.shape[1]
    }

def train_replica_model(data_dict, device='cuda:0', save_path='./replica_results'):
    """
    完全复制原始notebook的训练过程
    """
    print("\n=== 开始训练（完全复制原始notebook） ===")
    
    # 创建保存目录
    os.makedirs(save_path, exist_ok=True)
    
    # 模型配置（和原始notebook完全一致）
    batch_size = 128
    num_epochs = 25
    learning_rate = 1e-5  # 0.00001
    num_classes = 102
    
    print(f"训练配置:")
    print(f"  batch_size: {batch_size}")
    print(f"  num_epochs: {num_epochs}")
    print(f"  learning_rate: {learning_rate}")
    print(f"  num_classes: {num_classes}")
    
    # 创建数据集和数据加载器
    train_dataset = ExactReplicaDataset(data_dict['train_data'], data_dict['train_labels'])
    val_dataset = ExactReplicaDataset(data_dict['val_data'], data_dict['val_labels'])
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    
    print(f"数据加载器创建完成:")
    print(f"  训练批次数: {len(train_loader)}")
    print(f"  验证批次数: {len(val_loader)}")
    
    # 创建模型（完全复制原始架构）
    model = ReplicaMLPModel(
        input_dim=data_dict['feature_dim'],
        num_classes=num_classes,
        dropout_rate=0.5,
        l2_reg=1e-5  # 0.00001
    )
    model = model.to(device)
    
    print(f"模型参数数量: {sum(p.numel() for p in model.parameters())}")
    
    # 优化器（和原始notebook一致）
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    
    # 损失函数
    criterion = nn.CrossEntropyLoss()  # 用于one-hot标签
    
    # 训练记录
    train_losses = []
    train_accuracies = []
    val_losses = []
    val_accuracies = []
    val_f1_scores = []
    
    print(f"\n开始训练...")
    start_time = time.time()
    
    for epoch in range(num_epochs):
        # 训练阶段
        model.train()
        epoch_train_loss = 0.0
        epoch_train_acc = 0.0
        num_train_batches = 0
        
        for batch_idx, (data, target) in enumerate(train_loader):
            data, target = data.to(device), target.to(device)
            
            optimizer.zero_grad()
            output = model(data)
            
            # 计算损失（分类损失 + L2正则化）
            classification_loss = criterion(output, target)
            l2_loss = model.get_l2_loss()
            total_loss = classification_loss + l2_loss
            
            total_loss.backward()
            optimizer.step()
            
            # 统计
            epoch_train_loss += total_loss.item()
            pred = torch.argmax(output, dim=1)
            true = torch.argmax(target, dim=1)
            epoch_train_acc += (pred == true).float().mean().item()
            num_train_batches += 1
        
        # 计算平均训练指标
        avg_train_loss = epoch_train_loss / num_train_batches
        avg_train_acc = epoch_train_acc / num_train_batches
        train_losses.append(avg_train_loss)
        train_accuracies.append(avg_train_acc)
        
        # 验证阶段
        model.eval()
        epoch_val_loss = 0.0
        all_val_preds = []
        all_val_true = []
        num_val_batches = 0
        
        with torch.no_grad():
            for data, target in val_loader:
                data, target = data.to(device), target.to(device)
                output = model(data)
                
                classification_loss = criterion(output, target)
                l2_loss = model.get_l2_loss()
                total_loss = classification_loss + l2_loss
                
                epoch_val_loss += total_loss.item()
                
                pred = torch.argmax(output, dim=1)
                true = torch.argmax(target, dim=1)
                
                all_val_preds.extend(pred.cpu().numpy())
                all_val_true.extend(true.cpu().numpy())
                num_val_batches += 1
        
        # 计算验证指标
        avg_val_loss = epoch_val_loss / num_val_batches
        val_acc = accuracy_score(all_val_true, all_val_preds)
        val_f1 = f1_score(all_val_true, all_val_preds, average='macro')
        
        val_losses.append(avg_val_loss)
        val_accuracies.append(val_acc)
        val_f1_scores.append(val_f1)
        
        # 打印进度（和原始notebook类似）
        print(f"Epoch {epoch+1}/{num_epochs}:")
        print(f"  Train Loss: {avg_train_loss:.6f}, Train Acc: {avg_train_acc:.4f}")
        print(f"  Val Loss: {avg_val_loss:.6f}, Val Acc: {val_acc:.4f}, Val F1: {val_f1:.4f}")
        
        # 保存最佳模型
        if epoch == 0 or val_f1 > max(val_f1_scores[:-1]):
            best_model_path = os.path.join(save_path, 'best_replica_model.pth')
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_f1': val_f1,
                'val_acc': val_acc,
                'scaler': data_dict['scaler']
            }, best_model_path)
            print(f"  -> 保存最佳模型 (F1: {val_f1:.4f})")
    
    training_time = time.time() - start_time
    print(f"\n训练完成! 总耗时: {training_time:.2f}秒")
    
    # 最终评估
    best_f1 = max(val_f1_scores)
    best_epoch = val_f1_scores.index(best_f1) + 1
    best_acc = val_accuracies[val_f1_scores.index(best_f1)]
    
    print(f"\n=== 最终结果 ===")
    print(f"最佳验证F1分数: {best_f1:.4f} (第{best_epoch}轮)")
    print(f"对应验证准确率: {best_acc:.4f}")
    
    # 保存训练历史
    history = {
        'train_losses': train_losses,
        'train_accuracies': train_accuracies,
        'val_losses': val_losses,
        'val_accuracies': val_accuracies,
        'val_f1_scores': val_f1_scores,
        'best_f1': best_f1,
        'best_epoch': best_epoch,
        'training_time': training_time
    }
    
    history_path = os.path.join(save_path, 'training_history.npy')
    np.save(history_path, history)
    
    # 绘制训练曲线
    plot_training_curves(history, save_path)
    
    return history, model

def plot_training_curves(history, save_path):
    """绘制训练曲线"""
    plt.figure(figsize=(15, 5))
    
    # 损失曲线
    plt.subplot(1, 3, 1)
    plt.plot(history['train_losses'], label='Train Loss')
    plt.plot(history['val_losses'], label='Val Loss')
    plt.title('Training and Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    
    # 准确率曲线
    plt.subplot(1, 3, 2)
    plt.plot(history['train_accuracies'], label='Train Acc')
    plt.plot(history['val_accuracies'], label='Val Acc')
    plt.title('Training and Validation Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)
    
    # F1分数曲线
    plt.subplot(1, 3, 3)
    plt.plot(history['val_f1_scores'], label='Val F1', color='green')
    plt.title('Validation F1 Score')
    plt.xlabel('Epoch')
    plt.ylabel('F1 Score')
    plt.legend()
    plt.grid(True)
    
    plt.tight_layout()
    plt.savefig(os.path.join(save_path, 'training_curves.png'), dpi=300, bbox_inches='tight')
    plt.show()



In [ ]:

print("=== 完全复制原始brain_voxel.ipynb的训练 ===")

# 配置
mat_file_path = "/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/DATA/TRAIN38.mat"
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
save_path = './replica_training_results'

print(f"使用设备: {device}")
print(f"MAT文件: {mat_file_path}")
print(f"结果保存到: {save_path}")

# 检查文件是否存在
if not os.path.exists(mat_file_path):
    print(f"错误: MAT文件不存在: {mat_file_path}")
    return

try:
    # 1. 加载和处理数据（完全复制原始逻辑）
    data_dict = load_data_exact_replica(mat_file_path)
    
    # 2. 训练模型（完全复制原始逻辑）
    history, model = train_replica_model(data_dict, device, save_path)
    
    # 3. 和你现有的MAT格式结果对比
    print(f"\n=== 对比分析 ===")
    print(f"复制版本最佳F1: {history['best_f1']:.4f}")
    print(f"请与你现有的MAT格式训练结果进行对比")
    
    # 保存详细报告
    report_path = os.path.join(save_path, 'replica_training_report.txt')
    with open(report_path, 'w') as f:
        f.write("完全复制原始notebook的训练结果报告\n")
        f.write("="*50 + "\n\n")
        f.write(f"数据集信息:\n")
        f.write(f"  训练集: {data_dict['train_data'].shape}\n")
        f.write(f"  验证集: {data_dict['val_data'].shape}\n")
        f.write(f"  测试集: {data_dict['test_data'].shape}\n\n")
        f.write(f"训练结果:\n")
        f.write(f"  最佳验证F1: {history['best_f1']:.6f}\n")
        f.write(f"  最佳轮次: {history['best_epoch']}\n")
        f.write(f"  训练时间: {history['training_time']:.2f}秒\n\n")
        f.write(f"模型配置:\n")
        f.write(f"  架构: 4x4096 MLP\n")
        f.write(f"  Dropout: 0.5\n")
        f.write(f"  L2正则化: 1e-5\n")
        f.write(f"  学习率: 1e-5\n")
        f.write(f"  批大小: 128\n")
        f.write(f"  训练轮数: 25\n")
    
    print(f"详细报告保存到: {report_path}")
    print("训练完成!")
    
except Exception as e:
    print(f"训练过程中出错: {e}")
    import traceback
    traceback.print_exc()


In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

"""
基于复制版本，但将one-hot标签转换为索引标签的训练脚本
用于测试标签格式对模型性能的影响
"""

import os
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, accuracy_score, cohen_kappa_score, balanced_accuracy_score
from torch.utils.data import Dataset, DataLoader
import time
import sys

# 导入你现有的MAT加载函数
sys.path.append('.')
from data.mat_loader import load_mat_data

class IndexLabelDataset(Dataset):
    """
    使用索引标签的数据集类（而不是one-hot）
    """
    def __init__(self, data, labels, ignore_background=True):
        self.data = torch.FloatTensor(data)
        
        # 将one-hot标签转换为索引标签
        if len(labels.shape) > 1 and labels.shape[1] > 1:
            # 输入是one-hot格式，转换为索引
            self.labels = torch.LongTensor(np.argmax(labels, axis=1))
        else:
            # 已经是索引格式
            self.labels = torch.LongTensor(labels.astype(int))
        
        # 处理背景标签（如果需要）
        if ignore_background:
            # 将标签0（背景）转换为-1（忽略索引）
            # 将标签1-102转换为0-101
            background_mask = self.labels == 0
            self.labels = self.labels - 1  # 1-102 -> 0-101
            self.labels[background_mask] = -1  # 背景设为-1
        
        print(f"标签转换完成:")
        print(f"  标签形状: {self.labels.shape}")
        print(f"  标签范围: {torch.min(self.labels)} - {torch.max(self.labels)}")
        print(f"  背景标签(-1)数量: {torch.sum(self.labels == -1)}")
        print(f"  有效标签数量: {torch.sum(self.labels >= 0)}")
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]

class IndexLabelMLPModel(nn.Module):
    """
    用于索引标签的MLP模型（移除输出层的Softmax，因为CrossEntropyLoss已包含）
    """
    def __init__(self, input_dim=341, num_classes=102, dropout_rate=0.5, l2_reg=1e-5):
        super(IndexLabelMLPModel, self).__init__()
        
        self.layers = nn.Sequential(
            # 第一层：341 -> 4096
            nn.Linear(input_dim, 4096),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            # 第二层：4096 -> 4096  
            nn.Linear(4096, 4096),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            # 第三层：4096 -> 4096
            nn.Linear(4096, 4096), 
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            # 第四层：4096 -> 4096
            nn.Linear(4096, 4096),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            # 输出层：4096 -> 102 (注意：没有Softmax，因为CrossEntropyLoss会处理)
            nn.Linear(4096, num_classes)
        )
        
        # 应用L2正则化到所有线性层
        self.l2_reg = l2_reg
        for layer in self.layers:
            if isinstance(layer, nn.Linear):
                nn.init.xavier_uniform_(layer.weight)
                if layer.bias is not None:
                    nn.init.zeros_(layer.bias)
    
    def forward(self, x):
        return self.layers(x)
    
    def get_l2_loss(self):
        """计算L2正则化损失"""
        l2_loss = 0
        for layer in self.layers:
            if isinstance(layer, nn.Linear):
                l2_loss += torch.sum(layer.weight ** 2)
        return self.l2_reg * l2_loss

def load_data_with_index_labels(mat_file_path, random_state=42):
    """
    加载数据并转换为索引标签格式
    """
    print("=== 加载数据并转换为索引标签格式 ===")
    
    # 使用你现有的MAT加载函数
    print(f"加载MAT文件: {mat_file_path}")
    arrays = load_mat_data(mat_file_path)
    
    # 转置数据（和原始notebook一致）
    train_data = arrays['data'].transpose() if 'data' in arrays else arrays['data'].T
    train_region = arrays['region'].transpose() if 'region' in arrays else arrays['region'].T
    prob_idx = arrays['prob_idx'].transpose().flatten() if 'prob_idx' in arrays else arrays['prob_idx'].flatten()
    
    print(f"原始数据形状:")
    print(f"  train_data: {train_data.shape}")
    print(f"  train_region: {train_region.shape} (one-hot格式)")
    print(f"  prob_idx: {prob_idx.shape}")
    
    # 检查标签格式
    print(f"标签格式检查:")
    print(f"  是否为one-hot: {len(train_region.shape) > 1 and train_region.shape[1] > 1}")
    if len(train_region.shape) > 1 and train_region.shape[1] > 1:
        print(f"  每个样本标签和: {np.sum(train_region[0])}")  # 应该为1
        print(f"  标签维度: {train_region.shape[1]}")
    
    # 第一步：按患者分割
    curr_set = np.where(prob_idx != 38)[0]  # 患者1-37
    set_data = train_data[curr_set, :]
    set_region = train_region[curr_set, :]
    print(f"患者1-37数据形状: {set_data.shape}")
    
    curr_val = np.where(prob_idx == 38)[0]   # 患者38
    val_data = train_data[curr_val, :]
    val_label = train_region[curr_val, :]
    print(f"患者38数据形状: {val_data.shape}")
    
    # 第二步：从患者1-37中分出1%作为测试集
    X_train1, X_train2, y_train1, y_train2 = train_test_split(
        set_data, set_region, 
        test_size=0.01,
        random_state=42
    )
    
    print(f"数据分割结果:")
    print(f"  训练集: {X_train1.shape}")
    print(f"  测试集: {X_train2.shape}")
    print(f"  验证集: {val_data.shape}")
    
    # 第三步：标准化
    print("应用标准化...")
    scaler = StandardScaler()
    scaler.fit(X_train1)
    
    X_train1_scaled = scaler.transform(X_train1)
    X_train2_scaled = scaler.transform(X_train2)
    val_data_scaled = scaler.transform(val_data)
    
    print("标准化完成")
    
    return {
        'train_data': X_train1_scaled,
        'train_labels': y_train1,  # 保持one-hot格式，在Dataset中转换
        'test_data': X_train2_scaled, 
        'test_labels': y_train2,
        'val_data': val_data_scaled,
        'val_labels': val_label,
        'scaler': scaler,
        'feature_dim': X_train1_scaled.shape[1],
        'num_classes': y_train1.shape[1] if len(y_train1.shape) > 1 else int(np.max(y_train1)) + 1
    }

def calculate_class_weights(labels, num_classes=102, ignore_index=-1):
    """
    计算类别权重，处理不平衡问题
    """
    # 统计每个类别的样本数
    class_counts = np.bincount(labels[labels >= 0], minlength=num_classes)
    total_samples = len(labels[labels >= 0])  # 排除背景标签
    
    # 计算权重
    weights = np.zeros(num_classes, dtype=np.float32)
    for i in range(num_classes):
        if class_counts[i] > 0:
            weights[i] = total_samples / (num_classes * class_counts[i])
        else:
            weights[i] = 0.0
    
    print(f"类别权重计算完成:")
    print(f"  权重范围: {np.min(weights[weights > 0]):.4f} - {np.max(weights):.4f}")
    print(f"  零权重类别数: {np.sum(weights == 0)}")
    
    return torch.FloatTensor(weights)

def train_index_label_model(data_dict, device='cuda:0', save_path='./index_label_results'):
    """
    训练使用索引标签的模型
    """
    print("\n=== 开始训练（索引标签版本） ===")
    
    # 创建保存目录
    os.makedirs(save_path, exist_ok=True)
    
    # 模型配置
    batch_size = 128
    num_epochs = 25
    learning_rate = 1e-5
    num_classes = 102
    
    print(f"训练配置:")
    print(f"  batch_size: {batch_size}")
    print(f"  num_epochs: {num_epochs}")
    print(f"  learning_rate: {learning_rate}")
    print(f"  num_classes: {num_classes}")
    print(f"  标签格式: 索引标签（非one-hot）")
    
    # 创建数据集和数据加载器
    train_dataset = IndexLabelDataset(data_dict['train_data'], data_dict['train_labels'], ignore_background=True)
    val_dataset = IndexLabelDataset(data_dict['val_data'], data_dict['val_labels'], ignore_background=True)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    
    print(f"数据加载器创建完成:")
    print(f"  训练批次数: {len(train_loader)}")
    print(f"  验证批次数: {len(val_loader)}")
    
    # 计算类别权重
    train_labels_index = train_dataset.labels.numpy()
    class_weights = calculate_class_weights(train_labels_index, num_classes).to(device)
    
    # 创建模型
    model = IndexLabelMLPModel(
        input_dim=data_dict['feature_dim'],
        num_classes=num_classes,
        dropout_rate=0.5,
        l2_reg=1e-5
    )
    model = model.to(device)
    
    print(f"模型参数数量: {sum(p.numel() for p in model.parameters())}")
    
    # 优化器
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    
    # 损失函数（使用类别权重和忽略索引）
    criterion = nn.CrossEntropyLoss(weight=class_weights, ignore_index=-1)
    
    # 训练记录
    train_losses = []
    train_accuracies = []
    train_f1_scores = []
    val_losses = []
    val_accuracies = []
    val_f1_scores = []
    val_kappa_scores = []
    val_balanced_acc_scores = []
    
    print(f"\n开始训练...")
    start_time = time.time()
    
    for epoch in range(num_epochs):
        # 训练阶段
        model.train()
        epoch_train_loss = 0.0
        all_train_preds = []
        all_train_true = []
        num_train_batches = 0
        
        for batch_idx, (data, target) in enumerate(train_loader):
            data, target = data.to(device), target.to(device)
            
            optimizer.zero_grad()
            output = model(data)
            
            # 计算损失（分类损失 + L2正则化）
            classification_loss = criterion(output, target)
            l2_loss = model.get_l2_loss()
            total_loss = classification_loss + l2_loss
            
            total_loss.backward()
            optimizer.step()
            
            # 统计（只计算非背景像素）
            epoch_train_loss += total_loss.item()
            pred = torch.argmax(output, dim=1)
            
            # 只评估非背景像素
            valid_mask = target != -1
            if valid_mask.sum() > 0:
                all_train_preds.extend(pred[valid_mask].cpu().numpy())
                all_train_true.extend(target[valid_mask].cpu().numpy())
            
            num_train_batches += 1
        
        # 计算训练指标
        avg_train_loss = epoch_train_loss / num_train_batches
        if len(all_train_true) > 0:
            train_acc = accuracy_score(all_train_true, all_train_preds)
            train_f1 = f1_score(all_train_true, all_train_preds, average='macro')
        else:
            train_acc = 0.0
            train_f1 = 0.0
        
        train_losses.append(avg_train_loss)
        train_accuracies.append(train_acc)
        train_f1_scores.append(train_f1)
        
        # 验证阶段
        model.eval()
        epoch_val_loss = 0.0
        all_val_preds = []
        all_val_true = []
        num_val_batches = 0
        
        with torch.no_grad():
            for data, target in val_loader:
                data, target = data.to(device), target.to(device)
                output = model(data)
                
                classification_loss = criterion(output, target)
                l2_loss = model.get_l2_loss()
                total_loss = classification_loss + l2_loss
                
                epoch_val_loss += total_loss.item()
                
                pred = torch.argmax(output, dim=1)
                
                # 只评估非背景像素
                valid_mask = target != -1
                if valid_mask.sum() > 0:
                    all_val_preds.extend(pred[valid_mask].cpu().numpy())
                    all_val_true.extend(target[valid_mask].cpu().numpy())
                
                num_val_batches += 1
        
        # 计算验证指标
        avg_val_loss = epoch_val_loss / num_val_batches
        if len(all_val_true) > 0:
            val_acc = accuracy_score(all_val_true, all_val_preds)
            val_f1 = f1_score(all_val_true, all_val_preds, average='macro')
            val_kappa = cohen_kappa_score(all_val_true, all_val_preds)
            val_balanced_acc = balanced_accuracy_score(all_val_true, all_val_preds)
        else:
            val_acc = val_f1 = val_kappa = val_balanced_acc = 0.0
        
        val_losses.append(avg_val_loss)
        val_accuracies.append(val_acc)
        val_f1_scores.append(val_f1)
        val_kappa_scores.append(val_kappa)
        val_balanced_acc_scores.append(val_balanced_acc)
        
        # 打印进度
        print(f"Epoch {epoch+1}/{num_epochs}:")
        print(f"  Train Loss: {avg_train_loss:.6f}, Train Acc: {train_acc:.4f}, Train F1: {train_f1:.4f}")
        print(f"  Val Loss: {avg_val_loss:.6f}, Val Acc: {val_acc:.4f}, Val F1: {val_f1:.4f}")
        print(f"  Val Kappa: {val_kappa:.4f}, Val Balanced Acc: {val_balanced_acc:.4f}")
        
        # 保存最佳模型
        if epoch == 0 or val_f1 > max(val_f1_scores[:-1]):
            best_model_path = os.path.join(save_path, 'best_index_label_model.pth')
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_f1': val_f1,
                'val_acc': val_acc,
                'val_kappa': val_kappa,
                'val_balanced_acc': val_balanced_acc,
                'scaler': data_dict['scaler'],
                'class_weights': class_weights.cpu(),
                'model_config': {
                    'input_dim': data_dict['feature_dim'],
                    'num_classes': num_classes,
                    'dropout_rate': 0.5,
                    'l2_reg': 1e-5
                }
            }, best_model_path)
            print(f"  -> 保存最佳模型 (F1: {val_f1:.4f})")
    
    training_time = time.time() - start_time
    print(f"\n训练完成! 总耗时: {training_time:.2f}秒")
    
    # 最终评估
    best_f1 = max(val_f1_scores)
    best_epoch = val_f1_scores.index(best_f1) + 1
    best_acc = val_accuracies[val_f1_scores.index(best_f1)]
    best_kappa = val_kappa_scores[val_f1_scores.index(best_f1)]
    best_balanced_acc = val_balanced_acc_scores[val_f1_scores.index(best_f1)]
    
    print(f"\n=== 最终结果（索引标签版本） ===")
    print(f"最佳验证F1分数: {best_f1:.4f} (第{best_epoch}轮)")
    print(f"对应验证准确率: {best_acc:.4f}")
    print(f"对应Kappa系数: {best_kappa:.4f}")
    print(f"对应平衡准确率: {best_balanced_acc:.4f}")
    
    # 保存训练历史
    history = {
        'train_losses': train_losses,
        'train_accuracies': train_accuracies,
        'train_f1_scores': train_f1_scores,
        'val_losses': val_losses,
        'val_accuracies': val_accuracies,
        'val_f1_scores': val_f1_scores,
        'val_kappa_scores': val_kappa_scores,
        'val_balanced_acc_scores': val_balanced_acc_scores,
        'best_f1': best_f1,
        'best_epoch': best_epoch,
        'best_acc': best_acc,
        'best_kappa': best_kappa,
        'best_balanced_acc': best_balanced_acc,
        'training_time': training_time
    }
    
    history_path = os.path.join(save_path, 'index_label_training_history.npy')
    np.save(history_path, history)
    
    # 绘制训练曲线
    plot_index_training_curves(history, save_path)
    
    return history, model

def plot_index_training_curves(history, save_path):
    """绘制索引标签版本的训练曲线"""
    plt.figure(figsize=(20, 8))
    
    # 损失曲线
    plt.subplot(2, 4, 1)
    plt.plot(history['train_losses'], label='Train Loss')
    plt.plot(history['val_losses'], label='Val Loss')
    plt.title('Training and Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    
    # 准确率曲线
    plt.subplot(2, 4, 2)
    plt.plot(history['train_accuracies'], label='Train Acc')
    plt.plot(history['val_accuracies'], label='Val Acc')
    plt.title('Training and Validation Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)
    
    # F1分数曲线
    plt.subplot(2, 4, 3)
    plt.plot(history['train_f1_scores'], label='Train F1')
    plt.plot(history['val_f1_scores'], label='Val F1')
    plt.title('F1 Score')
    plt.xlabel('Epoch')
    plt.ylabel('F1 Score')
    plt.legend()
    plt.grid(True)
    
    # Kappa系数曲线
    plt.subplot(2, 4, 4)
    plt.plot(history['val_kappa_scores'], label='Val Kappa', color='purple')
    plt.title('Validation Kappa Score')
    plt.xlabel('Epoch')
    plt.ylabel('Kappa Score')
    plt.legend()
    plt.grid(True)
    
    # 平衡准确率曲线
    plt.subplot(2, 4, 5)
    plt.plot(history['val_balanced_acc_scores'], label='Val Balanced Acc', color='orange')
    plt.title('Validation Balanced Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Balanced Accuracy')
    plt.legend()
    plt.grid(True)
    
    # 训练集vs验证集F1对比
    plt.subplot(2, 4, 6)
    train_val_diff = [t - v for t, v in zip(history['train_f1_scores'], history['val_f1_scores'])]
    plt.plot(train_val_diff, label='Train F1 - Val F1', color='red')
    plt.title('Training vs Validation F1 Difference')
    plt.xlabel('Epoch')
    plt.ylabel('F1 Difference')
    plt.legend()
    plt.grid(True)
    plt.axhline(y=0, color='black', linestyle='--', alpha=0.5)
    
    # 所有验证指标汇总
    plt.subplot(2, 4, 7)
    plt.plot(history['val_accuracies'], label='Accuracy')
    plt.plot(history['val_f1_scores'], label='F1 Score')
    plt.plot(history['val_kappa_scores'], label='Kappa')
    plt.plot(history['val_balanced_acc_scores'], label='Balanced Acc')
    plt.title('All Validation Metrics')
    plt.xlabel('Epoch')
    plt.ylabel('Score')
    plt.legend()
    plt.grid(True)
    
    # 最佳性能汇总（柱状图）
    plt.subplot(2, 4, 8)
    metrics = ['Accuracy', 'F1 Score', 'Kappa', 'Balanced Acc']
    values = [history['best_acc'], history['best_f1'], 
              history['best_kappa'], history['best_balanced_acc']]
    bars = plt.bar(metrics, values)
    plt.title('Best Validation Metrics')
    plt.ylabel('Score')
    plt.xticks(rotation=45)
    
    # 在柱状图上添加数值标签
    for bar, value in zip(bars, values):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
                f'{value:.3f}', ha='center', va='bottom')
    
    plt.tight_layout()
    plt.savefig(os.path.join(save_path, 'index_label_training_curves.png'), dpi=300, bbox_inches='tight')
    plt.show()

def compare_with_replica_results(index_history, replica_results_path='./replica_training_results'):
    """
    对比索引标签版本和one-hot版本的结果
    """
    print(f"\n=== 对比分析：索引标签 vs One-hot标签 ===")
    
    try:
        # 尝试加载复制版本的结果
        replica_history_path = os.path.join(replica_results_path, 'training_history.npy')
        if os.path.exists(replica_history_path):
            replica_history = np.load(replica_history_path, allow_pickle=True).item()
            
            print(f"One-hot版本结果:")
            print(f"  最佳F1分数: {replica_history['best_f1']:.4f}")
            print(f"  最佳轮次: {replica_history['best_epoch']}")
            print(f"  训练时间: {replica_history['training_time']:.2f}秒")
            
            print(f"\n索引标签版本结果:")
            print(f"  最佳F1分数: {index_history['best_f1']:.4f}")
            print(f"  最佳轮次: {index_history['best_epoch']}")
            print(f"  训练时间: {index_history['training_time']:.2f}秒")
            
            # 计算差异
            f1_diff = index_history['best_f1'] - replica_history['best_f1']
            print(f"\n性能差异:")
            print(f"  F1分数差异: {f1_diff:+.4f}")
            if abs(f1_diff) < 0.01:
                print(f"  -> 差异很小，标签格式影响不大")
            elif f1_diff > 0:
                print(f"  -> 索引标签版本更好")
            else:
                print(f"  -> One-hot标签版本更好")
                
        else:
            print(f"未找到复制版本的结果文件: {replica_history_path}")
            print(f"请先运行复制版本的训练")
            
    except Exception as e:
        print(f"对比分析时出错: {e}")


In [ ]:

print("=== 索引标签版本的训练（基于复制版本） ===")

# 配置
mat_file_path = "/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/DATA/TRAIN38.mat"
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
save_path = './index_label_training_results'

print(f"使用设备: {device}")
print(f"MAT文件: {mat_file_path}")
print(f"结果保存到: {save_path}")
print(f"关键差异: 使用索引标签而不是one-hot标签")

# 检查文件是否存在
if not os.path.exists(mat_file_path):
    print(f"错误: MAT文件不存在: {mat_file_path}")
    return

try:
    # 1. 加载和处理数据
    data_dict = load_data_with_index_labels(mat_file_path)
    
    # 2. 训练模型
    history, model = train_index_label_model(data_dict, device, save_path)
    
    # 3. 对比分析
    compare_with_replica_results(history)
    
    # 保存详细报告
    report_path = os.path.join(save_path, 'index_label_training_report.txt')
    with open(report_path, 'w') as f:
        f.write("索引标签版本训练结果报告\n")
        f.write("="*50 + "\n\n")
        f.write(f"关键差异: 使用索引标签代替one-hot标签\n\n")
        f.write(f"数据集信息:\n")
        f.write(f"  训练集: {data_dict['train_data'].shape}\n")
        f.write(f"  验证集: {data_dict['val_data'].shape}\n")
        f.write(f"  测试集: {data_dict['test_data'].shape}\n\n")
        f.write(f"训练结果:\n")
        f.write(f"  最佳验证F1: {history['best_f1']:.6f}\n")
        f.write(f"  最佳验证准确率: {history['best_acc']:.6f}\n")
        f.write(f"  最佳Kappa系数: {history['best_kappa']:.6f}\n")
        f.write(f"  最佳平衡准确率: {history['best_balanced_acc']:.6f}\n")
        f.write(f"  最佳轮次: {history['best_epoch']}\n")
        f.write(f"  训练时间: {history['training_time']:.2f}秒\n\n")
        f.write(f"模型配置:\n")
        f.write(f"  架构: 4x4096 MLP\n")
        f.write(f"  Dropout: 0.5\n")
        f.write(f"  L2正则化: 1e-5\n")
        f.write(f"  学习率: 1e-5\n")
        f.write(f"  批大小: 128\n")
        f.write(f"  训练轮数: 25\n")
        f.write(f"  标签格式: 索引标签（0-101，背景-1）\n")
        f.write(f"  损失函数: CrossEntropyLoss with class weights and ignore_index=-1\n\n")
        f.write(f"关键改进:\n")
        f.write(f"  1. 使用索引标签而不是one-hot标签\n")
        f.write(f"  2. 添加类别权重处理不平衡问题\n")
        f.write(f"  3. 正确处理背景标签（ignore_index=-1）\n")
        f.write(f"  4. 移除输出层的Softmax（CrossEntropyLoss已包含）\n")
    
    print(f"详细报告保存到: {report_path}")
    print("训练完成!")
    
    # 4. 总结对比
    print(f"\n=== 最终总结 ===")
    print(f"这次训练的目的是测试标签格式对性能的影响:")
    print(f"1. 复制版本: 使用one-hot标签 + Softmax输出")
    print(f"2. 索引版本: 使用索引标签 + CrossEntropyLoss + 类别权重")
    print(f"")
    print(f"主要差异:")
    print(f"- 标签格式: one-hot vs 索引")
    print(f"- 背景处理: 可能包含背景 vs 明确忽略背景(-1)")
    print(f"- 类别权重: 无 vs 有（处理不平衡）")
    print(f"- 损失函数: 可能双重Softmax vs 标准CrossEntropyLoss")
    print(f"")
    print(f"请对比两个版本的F1分数，判断哪些因素对性能影响最大!")
    
except Exception as e:
    print(f"训练过程中出错: {e}")
    import traceback
    traceback.print_exc()
